# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/viki22uied/ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #2 — "The Content Performance Curve"** (health score peaks at 61-90 days, decays to a
low of 14 at 271-365 days, then "recovers" to 25.1 at 365+). The paper itself flags the caveat I'd
also raise: the "recovery" bucket is not a random sample of old pages, it is old pages **that
someone chose to refresh**. Health score isn't an independent outcome measured on a fixed
population followed over time — it's a cross-sectional snapshot split by current age bucket.

**Methodology question:** is the 365+ bucket's higher health score evidence that refreshing old
content *causes* recovery, or is it survivorship/selection bias — editors are more likely to
refresh a 365+ page that still has commercial value or existing backlinks, so the refreshed
365+ population is pre-selected toward pages likelier to perform well anyway? A within-page
before/after comparison (same content_id, health score 60 days before vs. 60 days after its own
refresh date) would separate "refresh helps" from "healthy old pages get refreshed."

**Finding #4 — "The Freshness Multiplier"** (365+ day content refreshed within 30 days shows a
3.2x health boost and 57x more impressions). The paper is honest that the 361+ growth-to-decline
ratio (283:1) is unstable (283 growing vs. 1 declining page), but the headline 3.2x/57x refresh
numbers get less of that same scrutiny in the pull-quote.

**Methodology question:** what is the comparison group for "57x more impressions" — refreshed
365+ pages vs. *all* unrefreshed 365+ pages, or vs. 365+ pages that were *candidates* for refresh
but not chosen? If refresh decisions aren't random (they usually track existing keyword demand
or seasonality), some of that 57x lift is confounded with whatever made the page worth refreshing
in the first place, not the refresh action alone. Reporting the refreshed-vs-candidate-pool gap
(not refreshed-vs-everything) would make the causal read safer.

Both findings are useful, directional signals for building a refresh-priority score (which is
exactly what my ML-08 model does) — but neither is a controlled experiment, and my model below
inherits the same "this is an association, not a proven causal lever" limitation.


In [1]:
print('Findings discussed in the markdown cell above -- no computation needed for this section.')

Findings discussed in the markdown cell above -- no computation needed for this section.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Setup:** same feature set and Random Forest hyperparameters as the reference pipeline / my
ML-08 model (`scripts/03_train_model.py`'s `MODEL_NUMERIC_FEATURES` + `MODEL_CATEGORICAL_FEATURES`,
label `is_declining_label`), on the full 30,000-row starter dataset. **Before:** a naive random
75/25 row split with no grouping (`train_test_split`, stratified on the label only).
**After:** the honest client-grouped 75/25 split (`GroupShuffleSplit` on `client_id`, the same
idea as ML-08's `client_holdout` split). Random seed 42 throughout.


In [2]:
import sys, json
sys.path.append('../../scripts')
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, PROCESSED_DIR, precision_at_k

RANDOM_SEED = 42
frame = pd.read_csv(PROCESSED_DIR / "refresh_feature_vector.csv")
y = frame["is_declining_label"].astype(int)

numeric = frame[[c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
cat = frame[[c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]].fillna("unknown").astype(str)
enc = pd.get_dummies(cat, prefix=cat.columns, dummy_na=False, dtype=float)
X = pd.concat([numeric.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)
groups = frame["client_id"].astype(str)

def run_split(train_idx, test_idx, label):
    rf = RandomForestClassifier(class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
                                 n_estimators=200, n_jobs=-1, random_state=RANDOM_SEED)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    probs = rf.predict_proba(X.iloc[test_idx])[:, 1]
    yte = y.iloc[test_idx]
    overlap = len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))
    return {
        "split": label,
        "auc": round(roc_auc_score(yte, probs), 4),
        "ap": round(average_precision_score(yte, probs), 4),
        "p_at_50": round(precision_at_k(yte, probs, 50), 4),
        "p_at_200": round(precision_at_k(yte, probs, 200), 4),
        "train_rows": int(len(train_idx)),
        "test_rows": int(len(test_idx)),
        "client_overlap_train_test": int(overlap),
    }

# BEFORE: naive random row split, no grouping -- same client can appear in train AND test
tr_idx, te_idx = train_test_split(np.arange(len(X)), test_size=0.25, random_state=RANDOM_SEED, stratify=y)
before = run_split(tr_idx, te_idx, "before_random_split")

# AFTER: client-grouped split -- zero client overlap between train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
tr_idx2, te_idx2 = next(gss.split(X, y, groups))
after = run_split(tr_idx2, te_idx2, "after_client_grouped_split")

comparison = pd.DataFrame([before, after]).set_index("split")
print(comparison)

import os
os.makedirs("../outputs", exist_ok=True)
with open("../outputs/w06_split_comparison.json", "w") as f:
    json.dump({"before_random_split": before, "after_client_grouped_split": after}, f, indent=2)

print()
print(f"Client overlap -- before: {before['client_overlap_train_test']} clients leaking across train/test; "
      f"after: {after['client_overlap_train_test']} (must be 0 for an honest grouped split)")
print(f"AUC drop from before to after: {before['auc'] - after['auc']:.4f}  "
      f"(the naive split's number was inflated by client memorization, not real skill)")


                               auc      ap  p_at_50  p_at_200  train_rows  \
split                                                                       
before_random_split         0.7598  0.7716     0.92     0.895       22500   
after_client_grouped_split  0.6058  0.5940     0.56     0.620       22885   

                            test_rows  client_overlap_train_test  
split                                                             
before_random_split              7500                         31  
after_client_grouped_split       7115                          0  

Client overlap -- before: 31 clients leaking across train/test; after: 0 (must be 0 for an honest grouped split)
AUC drop from before to after: 0.1540  (the naive split's number was inflated by client memorization, not real skill)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Checklist against `skills/hunting-leakage-and-validating/SKILL.md`'s taxonomy, run on the exact
feature list used above (`MODEL_NUMERIC_FEATURES` + `MODEL_CATEGORICAL_FEATURES` from
`scripts/ml_utils.py`, 52 columns after one-hot encoding):

- **Label-derived features:** the label `is_declining_label` is derived from `trend_direction`
  (itself derived from `trend_pct`). Both `trend_direction` and `trend_pct` are confirmed absent
  from the feature list (checked programmatically below) — this is the exact trap the GUIDE and
  ML-04 already called out, and it stays fixed here.
- **Future/overlapping windows:** all features are 90-day trailing aggregates or static content
  metadata (age, word count, position, CTR) computed over the same window the label's trend
  compares against its *own* prior 30 days — no feature reaches forward past the decision moment.
  `ctr` and `avg_position` are current-state facts, not outcomes computed after the label window.
- **Decision-derived / product-flag features:** none of `age_tier`, `position_tier`,
  `impression_tier`, `freshness_tier`, etc. are outputs of an existing recommendation system —
  they're bucketed versions of raw metrics (age, position, impressions), not someone else's
  triage decision. The reference pipeline's own `baseline_rules` score is correctly used only as
  a comparison baseline, never as a feature.
- **IDs as features:** `client_id` / `content_id` are used only for grouping the split, never as
  model inputs — confirmed below.
- **Top feature importance sanity check:** the top feature (`days_with_impressions`, ~0.158) does
  not dominate the way a leaked feature would (compare Week-4/ML-08's flagged case where one
  feature carried ~0.79 of total importance) — importance here is spread across 10+ features,
  which is a healthier sign than a single feature towering over the rest.

Verdict: no label-derived, future-window, or decision-derived leakage found in the current
feature set.


In [3]:
import sys
sys.path.append('../../scripts')
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES

feature_set = set(MODEL_NUMERIC_FEATURES) | set(MODEL_CATEGORICAL_FEATURES)
print(f"Total features in current model: {len(feature_set)}")

label_derived_columns = {"trend_direction", "trend_pct", "is_declining_label"}
leaked = feature_set & label_derived_columns
print(f"Label-derived columns found in feature set: {sorted(leaked) if leaked else 'NONE -- clean'}")

id_columns = {"client_id", "content_id"}
ids_as_features = feature_set & id_columns
print(f"Pseudonymous IDs used as features (should be none -- IDs are for grouping only): {sorted(ids_as_features) if ids_as_features else 'NONE -- clean'}")

decision_flag_columns = {"score", "action", "confidence"}  # reference pipeline's own output columns
decision_leak = feature_set & decision_flag_columns
print(f"Baseline/decision-system output columns used as features: {sorted(decision_leak) if decision_leak else 'NONE -- clean'}")

print()
print("Feature list:", sorted(feature_set))


Total features in current model: 26
Label-derived columns found in feature set: NONE -- clean
Pseudonymous IDs used as features (should be none -- IDs are for grouping only): NONE -- clean
Baseline/decision-system output columns used as features: NONE -- clean

Feature list: ['age_tier', 'ai_traffic_pct', 'avg_position', 'char_count', 'competition', 'competition_level', 'content_age_days', 'content_type', 'cpc', 'ctr', 'days_since_last_update', 'days_with_impressions', 'days_with_sessions', 'engagement_rate', 'freshness_tier', 'impression_tier', 'log_ai_sessions_90d', 'log_clicks_90d', 'log_impressions_90d', 'log_sessions_90d', 'main_intent', 'position_tier', 'scroll_rate', 'search_volume', 'word_count', 'word_count_tier']


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from ML-08 / `w05_model.ipynb`, section 4):** *"Random Forest wins on every metric
(AUC 0.846, AP 0.787, P@50 0.98, P@200 0.965) and beats the ML-07 baseline rule by a wide
margin."*

**Rewritten:** *"On a client-grouped holdout, Random Forest showed the strongest discrimination
of the three models compared (measured ROC AUC and top-K precision above the baseline rule and
above the other two candidates on this dataset and this split). This is an observed, in-sample
comparison on one build window — it is decision-support for prioritizing which pages a reviewer
looks at first, not a guarantee of performance on a different time window, client mix, or
population, and it does not establish that acting on the score causes any traffic outcome."*

Why the rewrite matters: "wins on every metric" and "beats ... by a wide margin" read as settled,
general facts. The honest version scopes the claim to what was actually measured (one split, one
window, one dataset) and states plainly what the number is for (ranking/triage support) and what
it is not (a causal or generalizable claim) — directly the language `work/README.md` and the
capstone checklist require.


In [4]:
print('See the markdown cell above for the rewritten claim -- no computation needed for this section, '
      'this is a language/communication check, not a numeric one.')


See the markdown cell above for the rewritten claim -- no computation needed for this section, this is a language/communication check, not a numeric one.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymized `client_id`/`content_id`
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
